<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/ai/02_machine_learning/supervised/regression/regression_model_pipeline_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score


class DataLoader:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None

    def load_data(self):
        self.data = pd.read_csv(self.file_path)
        return self.data

    def split_data(self, target_column, test_size=0.2):
        X = self.data.drop(columns=[target_column])
        y = self.data[target_column]
        return train_test_split(
            X, y, test_size=test_size, random_state=42
        )



class Preprocessor:
    def __init__(self):
        self.scaler = StandardScaler()

    def fit_transform(self, X_train):
        return self.scaler.fit_transform(X_train)

    def transform(self, X_test):
        return self.scaler.transform(X_test)




class BaseModel:
    def fit(self, X, y):
        raise NotImplementedError("fit() method not implemented")

    def predict(self, X):
        raise NotImplementedError("predict() method not implemented")

    def evaluate(self, X, y):
        predictions = self.predict(X)
        mse = mean_squared_error(y, predictions)
        r2 = r2_score(y, predictions)
        return mse, r2



class LinearRegressionModel(BaseModel):
    def __init__(self):
        self.model = LinearRegression()

    def fit(self, X, y):
        self.model.fit(X, y)

    def predict(self, X):
        return self.model.predict(X)



class DecisionTreeModel(BaseModel):
    def __init__(self, max_depth=5):
        self.model = DecisionTreeRegressor(max_depth=max_depth)

    def fit(self, X, y):
        self.model.fit(X, y)

    def predict(self, X):
        return self.model.predict(X)


class ModelTrainer:
    def __init__(self, models):
        self.models = models

    def train_and_evaluate(self, X_train, X_test, y_train, y_test):
        results = {}
        for model in self.models:
            model.fit(X_train, y_train)
            mse, r2 = model.evaluate(X_test, y_test)
            results[model.__class__.__name__] = {
                "MSE": mse,
                "R2_Score": r2
            }
        return results


if __name__ == "__main__":


    np.random.seed(42)
    data = pd.DataFrame({
        "Hours_Studied": np.random.randint(1, 10, 50),
        "Attendance": np.random.randint(60, 100, 50),
        "Marks": np.random.randint(40, 100, 50)
    })

    data.to_csv("student_data.csv", index=False)


    loader = DataLoader("student_data.csv")
    df = loader.load_data()

    X_train, X_test, y_train, y_test = loader.split_data("Marks")


    preprocessor = Preprocessor()
    X_train_scaled = preprocessor.fit_transform(X_train)
    X_test_scaled = preprocessor.transform(X_test)


    models = [
        LinearRegressionModel(),
        DecisionTreeModel(max_depth=4)
    ]


    trainer = ModelTrainer(models)
    results = trainer.train_and_evaluate(
        X_train_scaled, X_test_scaled, y_train, y_test
    )

    # ----------------------------
    # Display Results
    # ----------------------------
    for model_name, metrics in results.items():
        print(f"\nModel: {model_name}")
        print("Mean Squared Error:", metrics["MSE"])
        print("R2 Score:", metrics["R2_Score"])


Model: LinearRegressionModel
Mean Squared Error: 367.53213803934375
R2 Score: -0.1787432265533797

Model: DecisionTreeModel
Mean Squared Error: 432.0601851851852
R2 Score: -0.3856965528710239
